## Gold Layer — Gear Wear Tracking
Combines gear inventory with usage statistics to track wear progress and plan replacements.

**Purpose:** Single source of truth for gear lifecycle management (Q6: "Which gear is approaching its mileage limit?").

**Sources:**
* `silver.gear_list` - gear inventory with max distance limits
* `silver.gear_stats` - aggregated usage per gear item

**Key Metrics:**
* `wear_pct` - Percentage of max distance used (0-100+%)
* `km_remaining` - Distance left before reaching max limit (can be negative for overused gear)
* `is_active` - Whether gear is currently in rotation

**Use Cases:**
* Track which shoes are approaching replacement threshold
* Identify retired gear that exceeded its lifespan
* Plan gear purchases before current items wear out

In [0]:
%sql
CREATE OR REPLACE TABLE garmin_lakehouse.gold.gear_wear_tracking AS
SELECT
    -- Gear identification
    gl.gear_id,
    gl.gear_name,
    gl.gear_type,
    
    -- Status flag
    CASE WHEN gl.gear_status = 'active' THEN TRUE ELSE FALSE END as is_active,
    
    -- Usage metrics from gear_stats
    COALESCE(gs.total_distance_km, 0.0) as total_distance_km,
    
    -- Limit from gear_list
    gl.max_distance_km,
    
    -- Calculated wear metrics (Q6 headline metrics)
    CASE 
        WHEN gl.max_distance_km > 0 
        THEN ROUND((COALESCE(gs.total_distance_km, 0.0) / gl.max_distance_km) * 100, 1)
        ELSE 0.0
    END as wear_pct,
    
    ROUND(gl.max_distance_km - COALESCE(gs.total_distance_km, 0.0), 1) as km_remaining,
    
    -- Activity metrics
    COALESCE(gs.total_activities, 0) as total_activities,
    
    -- Timeline
    gs.updated_at as last_used,
    gl.date_start,
    
    -- Metadata
    CURRENT_TIMESTAMP() as loaded_at
    
FROM garmin_lakehouse.silver.gear_list gl
LEFT JOIN garmin_lakehouse.silver.gear_stats gs ON gl.gear_id = gs.gear_id

In [0]:
%sql
-- Preview: Active gear ordered by wear percentage (most worn first)
SELECT 
    gear_name,
    gear_type,
    is_active,
    total_distance_km,
    max_distance_km,
    wear_pct,
    km_remaining,
    total_activities,
    last_used,
    date_start
FROM garmin_lakehouse.gold.gear_wear_tracking
ORDER BY is_active DESC, wear_pct DESC